# بستن پوزیشن‌های باز — XAUUSD Order Block Bot
بستن دستی پوزیشن‌های باز ربات از MetaTrader 5 — magic `8088080`

**ترتیب اجرا:** سلول‌ها را از بالا به پایین اجرا کنید. MT5 باید باز و لاگین باشد.

---
- **سلول 3** — نمایش **همه** پوزیشن‌های باز (بدون فیلتر)
- **سلول 4** — بستن **همه** پوزیشن‌های ربات (magic=8088080)
- **سلول 5** — بستن **یک پوزیشن خاص** با ticket number
- **سلول 6** — تأیید نتیجه بعد از بستن
- **سلول 7** — قطع اتصال

## 1 — ایمپورت‌ها و تنظیمات

In [43]:
from __future__ import annotations
import os, sys
from datetime import datetime, timezone

import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 180)
pd.set_option('display.float_format', '{:.2f}'.format)

try:
    import MetaTrader5 as mt5
except ImportError:
    sys.exit('pip install MetaTrader5')

MAGIC       = 8088080
SYMBOL_BASE = 'XAUUSD'

## 2 — اتصال به MT5 و شناسایی نام سیمبول

In [44]:
def mt5_connect():
    kwargs = {}
    for key, env in [('login','MT5_LOGIN'), ('password','MT5_PASSWORD'), ('server','MT5_SERVER')]:
        v = os.environ.get(env)
        if v:
            kwargs[key] = int(v) if key == 'login' else v
    path = os.environ.get('MT5_TERMINAL_PATH')
    if path:
        kwargs['path'] = path
    if not mt5.initialize(**kwargs):
        raise RuntimeError(f'mt5.initialize failed: {mt5.last_error()}')

mt5_connect()
ti = mt5.terminal_info()
print(f'Connected : {ti.connected}  |  Build: {ti.build}')

ai = mt5.account_info()
print(f'Login     : {ai.login}  |  Server: {ai.server}')
print(f'Balance   : {ai.balance:,.2f}  |  Equity: {ai.equity:,.2f}  |  Profit: {ai.profit:+.2f}')

Connected : True  |  Build: 5833
Login     : 13399867  |  Server: ErranteSC-Demo
Balance   : 1,001.80  |  Equity: 1,001.80  |  Profit: +0.00


## 3 — نمایش همه پوزیشن‌های باز (بدون فیلتر)

In [45]:
# دریافت همه پوزیشن‌های باز بدون فیلتر symbol
all_pos = mt5.positions_get() or []

if not all_pos:
    print('هیچ پوزیشن بازی در اکانت وجود ندارد.')
else:
    rows = []
    for p in all_pos:
        is_buy    = p.type == mt5.POSITION_TYPE_BUY
        tick      = mt5.symbol_info_tick(p.symbol)
        cur_price = tick.bid if is_buy else tick.ask
        rows.append({
            'ticket'    : p.ticket,
            'symbol'    : p.symbol,
            'magic'     : p.magic,
            'is_bot'    : p.magic == MAGIC,
            'direction' : 'BUY' if is_buy else 'SELL',
            'volume'    : p.volume,
            'open_price': p.price_open,
            'cur_price' : cur_price,
            'SL'        : p.sl,
            'TP'        : p.tp,
            'profit'    : p.profit,
            'open_time' : datetime.fromtimestamp(p.time, tz=timezone.utc).strftime('%Y-%m-%d %H:%M'),
            'comment'   : p.comment,
        })
    df_pos = pd.DataFrame(rows)
    bot_pos = df_pos[df_pos['is_bot']]
    print(f'کل پوزیشن‌های باز اکانت : {len(df_pos)}')
    print(f'متعلق به ربات (magic={MAGIC}): {len(bot_pos)}')
    print(f'سود/زیان کل فعلی: {df_pos["profit"].sum():+.2f} $')
    print()
    display(df_pos)

هیچ پوزیشن بازی در اکانت وجود ندارد.


## 4 — بستن همه پوزیشن‌های ربات (magic=8088080)
> **هشدار:** این سلول تمام پوزیشن‌های باز با magic=8088080 را فوری می‌بندد.

In [46]:
all_pos = mt5.positions_get() or []
our_pos = [p for p in all_pos if p.magic == MAGIC]

if not our_pos:
    print(f'هیچ پوزیشن بازی با magic={MAGIC} وجود ندارد.')
else:
    print(f'{len(our_pos)} پوزیشن ربات یافت شد. در حال بستن...')
    print()
    for p in our_pos:
        is_buy      = p.type == mt5.POSITION_TYPE_BUY
        tick        = mt5.symbol_info_tick(p.symbol)
        close_type  = mt5.ORDER_TYPE_SELL if is_buy else mt5.ORDER_TYPE_BUY
        close_price = tick.bid if is_buy else tick.ask
        direction   = 'BUY' if is_buy else 'SELL'

        request = {
            'action'      : mt5.TRADE_ACTION_DEAL,
            'symbol'      : p.symbol,
            'volume'      : p.volume,
            'type'        : close_type,
            'position'    : p.ticket,
            'price'       : close_price,
            'deviation'   : 20,
            'magic'       : p.magic,
            'comment'     : 'manual_close',
            'type_time'   : mt5.ORDER_TIME_GTC,
            'type_filling': mt5.ORDER_FILLING_IOC,
        }

        result = mt5.order_send(request)
        if result.retcode == mt5.TRADE_RETCODE_DONE:
            print(f'[OK]   ticket={p.ticket} | {p.symbol} | {direction} | close_price={close_price:.2f} | profit={p.profit:+.2f}')
        else:
            print(f'[FAIL] ticket={p.ticket} | retcode={result.retcode} | {result.comment}')

هیچ پوزیشن بازی با magic=8088080 وجود ندارد.


## 5 — بستن یک پوزیشن خاص (با ticket number)
ticket number مورد نظر را از ستون `ticket` در سلول 3 کپی کنید.

In [47]:
TARGET_TICKET = 0  # <-- ticket number را اینجا وارد کنید

if TARGET_TICKET == 0:
    print('لطفاً TARGET_TICKET را تنظیم کنید.')
else:
    pos = mt5.positions_get(ticket=TARGET_TICKET)
    if not pos:
        print(f'پوزیشن با ticket {TARGET_TICKET} پیدا نشد یا قبلاً بسته شده.')
    else:
        p           = pos[0]
        is_buy      = p.type == mt5.POSITION_TYPE_BUY
        tick        = mt5.symbol_info_tick(p.symbol)
        close_type  = mt5.ORDER_TYPE_SELL if is_buy else mt5.ORDER_TYPE_BUY
        close_price = tick.bid if is_buy else tick.ask
        direction   = 'BUY' if is_buy else 'SELL'

        print(f'در حال بستن: ticket={p.ticket} | {p.symbol} | {direction} | vol={p.volume} | profit={p.profit:+.2f}')

        request = {
            'action'      : mt5.TRADE_ACTION_DEAL,
            'symbol'      : p.symbol,
            'volume'      : p.volume,
            'type'        : close_type,
            'position'    : p.ticket,
            'price'       : close_price,
            'deviation'   : 20,
            'magic'       : p.magic,
            'comment'     : 'manual_close',
            'type_time'   : mt5.ORDER_TIME_GTC,
            'type_filling': mt5.ORDER_FILLING_IOC,
        }

        result = mt5.order_send(request)
        if result.retcode == mt5.TRADE_RETCODE_DONE:
            print(f'[OK]  بسته شد با قیمت {close_price:.2f}')
        else:
            print(f'[FAIL] retcode={result.retcode} | {result.comment}')

لطفاً TARGET_TICKET را تنظیم کنید.


## 6 — تأیید: وضعیت پوزیشن‌ها بعد از بستن

In [48]:
all_pos = mt5.positions_get() or []
our_pos = [p for p in all_pos if p.magic == MAGIC]

ai = mt5.account_info()
print(f'Balance : {ai.balance:,.2f}  |  Equity: {ai.equity:,.2f}  |  Profit: {ai.profit:+.2f}')
print()

if not our_pos:
    print('همه پوزیشن‌های ربات بسته شدند.')
else:
    print(f'هنوز {len(our_pos)} پوزیشن ربات باز وجود دارد:')
    rows = []
    for p in our_pos:
        rows.append({
            'ticket'   : p.ticket,
            'symbol'   : p.symbol,
            'direction': 'BUY' if p.type == mt5.POSITION_TYPE_BUY else 'SELL',
            'volume'   : p.volume,
            'profit'   : p.profit,
            'comment'  : p.comment,
        })
    display(pd.DataFrame(rows))

Balance : 1,001.80  |  Equity: 1,001.80  |  Profit: +0.00

همه پوزیشن‌های ربات بسته شدند.


## 7 — قطع اتصال از MT5

In [49]:
mt5.shutdown()
print('MT5 disconnected.')

MT5 disconnected.
